# Diagnóstico: Gap de alertas desde 24/08

**Propósito:** Investigar por que a tabela `observability.alertas` não recebeu novos registros
desde 24/08, determinando se a causa é falha de execução do `job_diario` ou ausência real de
condições de alerta desde então.

**Tipo:** Investigação pontual (não é teste automatizado — não roda em Job, não faz parte do CI).

**Tabelas envolvidas:**
- `poc_pulse_observability.observability.pipeline_runs`
- `poc_pulse_observability.observability.alertas`

**Resultado esperado:** Confirmação factual da causa raiz. Se um problema real for confirmado,
vira entrada em `docs/licoes-aprendidas.md` — este notebook é só o rastro de como se chegou lá.

**Autor:** Bruno Queles
**Data da investigação:** 2026-09-15

In [0]:
df_runs = spark.table("poc_pulse_observability.observability.pipeline_runs")
df_runs.printSchema()
df_runs.limit(5).display()

In [0]:
from pyspark.sql import functions as F

df_diario = (
    df_runs
    .withColumn("data_execucao", F.to_date("timestamp_execucao"))
    .filter(F.col("data_execucao") >= "2026-08-24")
    .groupBy("data_execucao", "status")
    .count()
    .orderBy("data_execucao", "status")
)

df_diario.display()

In [0]:
df_cadeia_fria = spark.table("poc_pulse_observability.observability.observability_cadeia_fria")
df_cadeia_fria.printSchema()
df_cadeia_fria.limit(10).display()

In [0]:
from pyspark.sql import functions as F

df_violacoes = (
    df_cadeia_fria
    .withColumn("data_remessa", F.to_date(F.substring("remessa_id", 5, 8), "yyyyMMdd"))
    .filter(~F.col("tipo_violacao").isin("nao_aplicavel", "conforme"))
)

df_violacoes.groupBy(
    F.col("data_remessa"),
    "tipo_violacao"
).count().orderBy("data_remessa", ascending=False).display()

In [0]:
# 1) Quando essa tabela foi escrita pela última vez de verdade (não o dado, o job)
spark.sql("DESCRIBE HISTORY poc_pulse_observability.observability.observability_cadeia_fria").select(
    "version", "timestamp", "operation"
).orderBy(F.col("version").desc()).limit(10).display()

In [0]:
# 2) O pipeline_runs tem alguma entrada específica pra essa tabela desde 21/08?
spark.table("poc_pulse_observability.observability.pipeline_runs") \
    .filter(F.col("item").rlike("(?i)cadeia|otif|temperatura")) \
    .withColumn("data_execucao", F.to_date("timestamp_execucao")) \
    .filter(F.col("data_execucao") >= "2026-08-15") \
    .select("data_execucao", "pipeline", "item", "status") \
    .orderBy("data_execucao", ascending=False) \
    .display()

In [0]:
df_cadeia_fria \
    .withColumn("data_remessa", F.to_date(F.substring("remessa_id", 5, 8), "yyyyMMdd")) \
    .groupBy("data_remessa") \
    .count() \
    .orderBy(F.col("data_remessa").desc()) \
    .limit(15) \
    .display()

In [0]:
df_remessas_silver = spark.table("poc_pulse_observability.silver.tms_remessas")

df_remessas_silver \
    .withColumn("data_expedicao", F.to_date("data_expedicao")) \
    .groupBy("data_expedicao") \
    .count() \
    .orderBy(F.col("data_expedicao").desc()) \
    .limit(10) \
    .display()

In [0]:
spark.table("poc_pulse_observability.observability.pipeline_runs") \
    .filter(F.col("item") == "tms_remessas") \
    .withColumn("data_execucao", F.to_date("timestamp_execucao")) \
    .filter(F.col("data_execucao") >= "2026-08-15") \
    .select("data_execucao", "pipeline", "status", "detalhes") \
    .orderBy("data_execucao", ascending=False) \
    .display()

In [0]:
spark.table("poc_pulse_observability.observability.pipeline_runs") \
    .filter(F.col("item") == "tms_remessas") \
    .filter(F.col("pipeline") != "ingerir_dados") \
    .withColumn("data_execucao", F.to_date("timestamp_execucao")) \
    .filter(F.col("data_execucao") >= "2026-08-15") \
    .select("data_execucao", "pipeline", "status", "detalhes") \
    .orderBy("data_execucao", ascending=False) \
    .display()

In [0]:
%sh
grep -rn "transformar_bronze_para_silver(" /Workspace/Users/bruno.quelestech@outlook.com/poc-pulse-health-observability/src

In [0]:
%sh
cat /Workspace/Users/bruno.quelestech@outlook.com/poc-pulse-health-observability/resources/job_diario.yml

In [0]:
%sh
grep -rln "transformar_bronze_para_silver" /Workspace/Users/bruno.quelestech@outlook.com/poc-pulse-health-observability --include="*.yml"

In [0]:
%sh
cat /Workspace/Users/bruno.quelestech@outlook.com/poc-pulse-health-observability/databricks.yml